# Search and Export of Sentinel-1 and Sentinel-2 Images

## DATA EXPORT: Sentinel-1 SAR + Sentinel-2 MSI
Search, selection and export of bitemporal images for use in the notebook
Change Detection via U-Net (`02.U-net_extraction_Change_Detection_Map_for_Validation.ipynb`)

**Satélites:** Sentinel-1 (SAR - VV, VH) | Sentinel-2 (MSI - Óptico)

**Versão:** 1.0 | **Data:** 2025

---
## 0.0 MOUNTING THE GOOGLE DRIVE
---

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


---
## 0.1 LIBRARY INSTALLATION
---

In [2]:
!pip install localtileserver==0.10.7 geemap==0.37.2 eemont==2025.7.1 \
  geopandas==1.1.2 rasterio==1.5.0 folium==0.20.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.6/184.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.7/341.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.4/281.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 63.0 MB/s eta 0:00:00


---
# 1.0 IMPORTING LIBRARIES
---

In [3]:
import ee
import geemap
import eemont
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas importadas com sucesso')

Bibliotecas importadas com sucesso


---
# 2.0 AUTENTICAÇÃO E INICIALIZAÇÃO DO GOOGLE EARTH ENGINE
---

In [4]:
try:
    ee.Initialize(project="ee-joaovictornh01") # put your geemap project
    print("Earth Engine ja autenticado")
except:
    print("Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project="ee-joaovictornh01") # put your geemap project
    print("Earth Engine autenticado com sucesso")

Autenticando Earth Engine...
Earth Engine autenticado com sucesso


---
## 3.1 INITIAL SETUP
---

In [5]:
def create_sentinel_config(
    start_date,
    end_date,
    geometry,
    cloud_cover=10,
    sentinel2_id=None,
    export_folder=None,
    export_crs='EPSG:32724'
):
    """
    Configura os parâmetros para busca de imagens Sentinel-2.

    Args:
        start_date (str): Data inicial (YYYY-MM-DD)
        end_date (str): Data final (YYYY-MM-DD)
        geometry: Geometria da região de interesse
        cloud_cover (int): Cobertura máxima de nuvens (%)
        sentinel2_id (str, optional): ID específico da imagem Sentinel-2
        export_folder (str): Pasta para exportação no Google Drive
        export_crs (str): Sistema de coordenadas para exportação
    """
    config = {
        'start_date': start_date,
        'end_date': end_date,
        'geometry': geometry,
        'cloud_cover': cloud_cover,
        'sentinel2_id': sentinel2_id,
        'export_folder': export_folder,
        'export_crs': export_crs
    }

    print('=' * 80)
    print('CONFIGURACAO SENTINEL')
    print('=' * 80)
    print(f"Periodo de busca: {start_date} a {end_date}")
    print(f"Cobertura de nuvens maxima: {cloud_cover}%")
    if sentinel2_id:
        print(f"ID Sentinel-2 selecionado: {sentinel2_id}")
    if export_folder:
        print(f"Pasta de exportacao: {export_folder}")
    print(f"Sistema de coordenadas: {export_crs}")
    print('=' * 80)

    return config


In [6]:
def create_bitemporal_sentinel_config(
    t1_start_date,
    t1_end_date,
    t2_start_date,
    t2_end_date,
    geometry,
    cloud_cover=10,
    sentinel2_t1_id=None,
    sentinel2_t2_id=None,
    export_folder=None,
    site_name='study_area',
    export_crs='EPSG:32724'
):
    """
    Configuração para busca e exportação bitemporal de imagens Sentinel.
    """
    config = {
        't1_start_date': t1_start_date,
        't1_end_date': t1_end_date,
        't2_start_date': t2_start_date,
        't2_end_date': t2_end_date,
        'geometry': geometry,
        'cloud_cover': cloud_cover,
        'sentinel2_t1_id': sentinel2_t1_id,
        'sentinel2_t2_id': sentinel2_t2_id,
        'export_folder': export_folder,
        'site_name': site_name,
        'export_crs': export_crs
    }

    print('=' * 80)
    print('CONFIGURACAO BITEMPORAL SENTINEL')
    print('=' * 80)
    print(f"Periodo T1: {t1_start_date} a {t1_end_date}")
    if sentinel2_t1_id:
        print(f"  Sentinel-2 T1 ID: {sentinel2_t1_id}")
    print(f"Periodo T2: {t2_start_date} a {t2_end_date}")
    if sentinel2_t2_id:
        print(f"  Sentinel-2 T2 ID: {sentinel2_t2_id}")
    print(f"Cobertura de nuvens maxima: {cloud_cover}%")
    print(f"Site: {site_name}")
    print(f"Sistema de coordenadas: {export_crs}")
    if export_folder:
        print(f"Pasta de exportacao: {export_folder}")
    print('=' * 80)

    return config


---
## 3.2 PROCESSING FUNCTIONS
---

In [7]:
def get_utm_crs(lon, lat):
    """Determina CRS UTM baseado em coordenadas."""
    zone_number = int((np.floor((lon + 180) / 6) % 60) + 1)
    hemisphere = '6' if lat >= 0 else '7'
    return f'EPSG:32{hemisphere}{zone_number:02d}'


In [8]:
def apply_sentinel2_scale_factors(image):
    """
    Aplica fatores de escala para Sentinel-2 (Level-2A / HARMONIZED).
    Bandas B2, B3, B4, B8, B11, B12 multiplicadas por 0.0001.
    """
    optical = (image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
               .multiply(0.0001))
    return image.addBands(optical, None, True)


In [9]:
def get_sentinel2_collection(config):
    """
    Busca coleção de imagens Sentinel-2 HARMONIZED.

    Nota: COPERNICUS/S2_HARMONIZED disponivel a partir de 2015-06-23.
          Para Surface Reflectance (S2_SR_HARMONIZED) apenas a partir de 2017-03-28.
    """
    collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
                 .filterBounds(config['geometry'])
                 .filterDate(config['start_date'], config['end_date'])
                 .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', config['cloud_cover'])
                 .map(apply_sentinel2_scale_factors))
    return collection


In [10]:
def get_sentinel1_collection(roi, start_date, end_date, polarizations=['VV', 'VH']):
    """
    Busca coleção de imagens Sentinel-1 SAR.

    Args:
        roi: Geometria da região de interesse
        start_date (str): Data inicial (YYYY-MM-DD)
        end_date (str): Data final (YYYY-MM-DD)
        polarizations (list): Polarizações a selecionar ['VV', 'VH']
    """
    collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(roi)
                  .filterDate(start_date, end_date)
                  .filterMetadata('instrumentMode', 'equals', 'IW')
                  .select(polarizations)
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))
    return collection


In [11]:
def process_sentinel2_median(sentinel2_collection):
    """
    Calcula a mediana da coleção Sentinel-2 (composto livre de nuvens).
    """
    print('Processando Sentinel-2 (mediana)...')
    median = sentinel2_collection.median()
    s2_bands = median.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
    s2_projection = median.select('B2').projection()
    print('  Mediana Sentinel-2 calculada')
    return s2_bands, s2_projection


In [12]:
def get_selected_sentinel2_image(sentinel2_id, roi):
    """
    Obtém e processa imagem Sentinel-2 específica por ID completo.

    Args:
        sentinel2_id (str): ID da imagem (ex: '20230101T131242_20230101T131242_T24MUA')
        roi: Região de interesse para recorte
    """
    s2_img = ee.Image(f'COPERNICUS/S2_HARMONIZED/{sentinel2_id}')
    s2_img = apply_sentinel2_scale_factors(s2_img).clip(roi)
    return s2_img


In [13]:
def process_sentinel2_image(sentinel2_image):
    """
    Extrai bandas (B2, B3, B4, B8, B11, B12) e projeção de imagem Sentinel-2.
    """
    print('Processando Sentinel-2 (imagem especifica)...')
    s2_bands = sentinel2_image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
    s2_projection = sentinel2_image.select('B2').projection()
    print('  Imagem Sentinel-2 processada')
    return s2_bands, s2_projection


In [14]:
def generate_sentinel1_composite_from_s2_date(roi, s2_image_id, polarizations=['VV', 'VH']):
    """
    Gera composição Sentinel-1 baseada na data da imagem Sentinel-2.
    Usa janela de +/- 30 dias em torno da data do S2 e seleciona
    a órbita (ascendente ou descendente) com mais imagens disponíveis.

    Args:
        roi: Geometria da região de interesse
        s2_image_id (str): ID completo da imagem Sentinel-2
        polarizations (list): Polarizações a selecionar

    Returns:
        ee.Image: Composição SAR normalizada no intervalo [0, 1]
    """
    product_id = s2_image_id

    if product_id and len(product_id) >= 8:
        date_part = product_id.split('_')[0]
        if len(date_part) >= 8:
            ano = date_part[:4]
            mes = date_part[4:6]
            dia = date_part[6:8]
            date_str = f'{ano}-{mes}-{dia}'
            s2_date = ee.Date(date_str)
        else:
            print(f'Erro: Nao foi possivel extrair data de: {date_part}')
            return None
    else:
        print(f'Erro: ID invalido: {product_id}')
        return None

    start_date = s2_date.advance(-30, 'day')
    end_date = s2_date.advance(30, 'day')

    print(f'Processando Sentinel-1 SAR para {date_str}...')
    start_str = start_date.format('YYYY-MM-dd').getInfo()
    end_str = end_date.format('YYYY-MM-dd').getInfo()
    print(f'  Janela temporal: {start_str} a {end_str}')

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(roi)
          .filterDate(start_date, end_date)
          .filterMetadata('instrumentMode', 'equals', 'IW')
          .select(polarizations))

    for pol in polarizations:
        s1 = s1.filter(ee.Filter.listContains('transmitterReceiverPolarisation', pol))

    s1 = s1.map(lambda img: img.updateMask(img.gte(-25)))

    asc = s1.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    desc = s1.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    asc_count = asc.size().getInfo()
    desc_count = desc.size().getInfo()

    print(f'  Imagens ascendentes: {asc_count} | descendentes: {desc_count}')

    s1 = asc if asc_count > desc_count else desc
    orbit_type = 'ASCENDING' if asc_count > desc_count else 'DESCENDING'
    print(f'  Orbita selecionada: {orbit_type}')

    orbit_numbers = (s1.toList(s1.size())
                     .map(lambda img: ee.Image(img).get('relativeOrbitNumber_start'))
                     .distinct())
    orbit_list = orbit_numbers.getInfo()
    print(f'  Orbitas relativas: {orbit_list}')

    def process_orbit(orbit_num):
        orbit_col = s1.filter(ee.Filter.eq('relativeOrbitNumber_start', orbit_num))
        return orbit_col.mean()

    means = ee.ImageCollection([process_orbit(o) for o in orbit_list])

    mean_mosaic = (means.mosaic()
                   .unitScale(-25, 0)
                   .clamp(0, 1)
                   .unmask()
                   .float())

    band_names = mean_mosaic.bandNames().getInfo()
    new_band_names = []
    for name in band_names:
        if 'VV' in name:
            new_band_names.append('VV')
        elif 'VH' in name:
            new_band_names.append('VH')
        else:
            new_band_names.append(name)
    mean_mosaic = mean_mosaic.rename(new_band_names)

    print('  Composicao SAR criada com sucesso')
    return mean_mosaic


---
## 3.3 SEARCH AND VIEW IMAGES
---

In [15]:
def search_available_sentinel2_images(config, max_display=20):
    """
    Busca e lista imagens Sentinel-2 disponíveis no período e área definidos.

    Args:
        config (dict): Dicionário de configuração
        max_display (int): Número máximo de imagens a exibir

    Returns:
        list: Lista de dicionários com id, date, cloud_cover
    """
    print('\n' + '=' * 80)
    print(f"BUSCANDO IMAGENS SENTINEL-2: {config['start_date']} a {config['end_date']}")
    print('=' * 80)

    s2_col = get_sentinel2_collection(config)
    n_s2 = s2_col.size().getInfo()
    print(f'Imagens encontradas: {n_s2}')

    s2_info = []
    if n_s2 > 0:
        print('\n--- IDs das imagens Sentinel-2 ---')
        s2_list = s2_col.toList(n_s2)
        for i in range(min(max_display, n_s2)):
            img = ee.Image(s2_list.get(i))
            img_id = img.id().getInfo()
            cloud_pct = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
            date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
            s2_info.append({'id': img_id, 'date': date, 'cloud_cover': cloud_pct})
            print(f'  {i+1}. {img_id}')
            print(f'     Data: {date} | Nuvens: {cloud_pct:.2f}%')
        if n_s2 > max_display:
            print(f'  ... e mais {n_s2 - max_display} imagens')
    else:
        print('AVISO: Nenhuma imagem Sentinel-2 encontrada!')
        print('  - Verifique o periodo de busca')
        print('  - Aumente o limite de cobertura de nuvens')
        print('  - Verifique se a geometria esta correta')

    return s2_info


In [16]:
def create_preview_map(roi, sentinel2_id, zoom=15):
    """
    Cria mapa interativo para visualizar imagem Sentinel-2 e o
    composito Sentinel-1 correspondente (+/- 30 dias).

    Args:
        roi: Região de interesse
        sentinel2_id (str): ID completo da imagem Sentinel-2
        zoom (int): Nível de zoom inicial

    Returns:
        geemap.Map
    """
    print('\n' + '=' * 80)
    print('GERANDO MAPA DE VISUALIZACAO')
    print('=' * 80)

    Map = geemap.Map()
    Map.centerObject(roi, zoom=zoom)
    Map.addLayer(roi, {'color': 'yellow'}, 'ROI', True, 0.5)

    # Sentinel-2
    print(f'Carregando Sentinel-2: {sentinel2_id}')
    try:
        s2_img = get_selected_sentinel2_image(sentinel2_id, roi)
        Map.addLayer(
            s2_img,
            {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3, 'gamma': 1.3},
            'Sentinel-2 RGB', True
        )
        Map.addLayer(
            s2_img,
            {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 0.3, 'gamma': 1.3},
            'Sentinel-2 Falsa Cor (NIR)', False
        )
        print('  Sentinel-2 carregado')
    except Exception as e:
        print(f'  Erro ao carregar Sentinel-2: {str(e)}')

    # Sentinel-1
    print(f'Carregando Sentinel-1 correspondente...')
    try:
        s1_composite = generate_sentinel1_composite_from_s2_date(roi, sentinel2_id)
        if s1_composite:
            Map.addLayer(
                s1_composite.select('VV').clip(roi),
                {'min': 0, 'max': 1, 'palette': ['black', 'white']},
                'Sentinel-1 VV', False
            )
            Map.addLayer(
                s1_composite.select('VH').clip(roi),
                {'min': 0, 'max': 1, 'palette': ['black', 'white']},
                'Sentinel-1 VH', False
            )
            s1_rgb = s1_composite.select(['VV', 'VH']).addBands(
                s1_composite.select('VV').divide(s1_composite.select('VH')).rename('ratio')
            )
            Map.addLayer(
                s1_rgb.clip(roi),
                {'bands': ['VV', 'VH', 'ratio'], 'min': [0, 0, 0], 'max': [1, 1, 2]},
                'Sentinel-1 RGB (VV/VH/ratio)', False
            )
            print('  Sentinel-1 carregado')
    except Exception as e:
        print(f'  Erro ao carregar Sentinel-1: {str(e)}')

    Map.add_layer_control()
    print('Mapa criado com sucesso')
    return Map


---
## 3.4 EXPORT FUNCTIONS
---

In [17]:
def export_sentinel2_bitemporal_data(roi, s2_t1, s2_t2, config, export_scale=10):
    """
    Exporta imagens Sentinel-2 bitemporais para o Google Drive.
    Exporta bandas: B2 (Blue), B3 (Green), B4 (Red), B8 (NIR), B11 (SWIR1), B12 (SWIR2).

    Args:
        roi: Região de interesse
        s2_t1 (ee.Image): Imagem Sentinel-2 período T1
        s2_t2 (ee.Image): Imagem Sentinel-2 período T2
        config (dict): Configuração com site_name, export_folder, export_crs
        export_scale (int): Resolução de exportação em metros (padrão 10m)

    Returns:
        list: Lista de tarefas EE iniciadas
    """
    print('\n' + '=' * 80)
    print('EXPORTANDO SENTINEL-2 PARA GOOGLE DRIVE')
    print('=' * 80)

    try:
        coords = roi.centroid(0.1).coordinates().getInfo()
        crs = get_utm_crs(coords[0], coords[1])
    except Exception:
        crs = config.get('export_crs', 'EPSG:4326')
    print(f'CRS: {crs} | Escala: {export_scale}m')

    bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
    tasks = []

    task_t1 = ee.batch.Export.image.toDrive(
        image=s2_t1.select(bands),
        description=f"Sentinel2_{config['site_name']}_T1",
        folder=config['export_folder'],
        fileNamePrefix=f"sentinel2_{config['site_name']}_t1",
        region=roi,
        crs=crs,
        scale=export_scale,
        maxPixels=1e12,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task_t1.start()
    tasks.append(task_t1)
    print(f"Exportacao Sentinel-2 T1 iniciada: sentinel2_{config['site_name']}_t1")

    task_t2 = ee.batch.Export.image.toDrive(
        image=s2_t2.select(bands),
        description=f"Sentinel2_{config['site_name']}_T2",
        folder=config['export_folder'],
        fileNamePrefix=f"sentinel2_{config['site_name']}_t2",
        region=roi,
        crs=crs,
        scale=export_scale,
        maxPixels=1e12,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task_t2.start()
    tasks.append(task_t2)
    print(f"Exportacao Sentinel-2 T2 iniciada: sentinel2_{config['site_name']}_t2")

    print(f"\nPasta de destino: {config['export_folder']}")
    return tasks


In [18]:
def export_sentinel1_bitemporal_data(roi, s1_t1, s1_t2, config, polarizations=['VV', 'VH'], export_scale=10):
    """
    Exporta composições Sentinel-1 SAR bitemporais para o Google Drive.
    Os valores VV e VH estão normalizados no intervalo [0, 1].

    Args:
        roi: Região de interesse
        s1_t1 (ee.Image): Composição SAR período T1
        s1_t2 (ee.Image): Composição SAR período T2
        config (dict): Configuração com site_name, export_folder, export_crs
        polarizations (list): Polarizações a exportar ['VV', 'VH']
        export_scale (int): Resolução de exportação em metros (padrão 10m)

    Returns:
        list: Lista de tarefas EE iniciadas
    """
    print('\n' + '=' * 80)
    print('EXPORTANDO SENTINEL-1 PARA GOOGLE DRIVE')
    print('=' * 80)

    try:
        coords = roi.centroid(0.1).coordinates().getInfo()
        crs = get_utm_crs(coords[0], coords[1])
    except Exception:
        crs = config.get('export_crs', 'EPSG:4326')
    print(f'CRS: {crs} | Polarizacoes: {polarizations} | Escala: {export_scale}m')

    tasks = []

    task_t1 = ee.batch.Export.image.toDrive(
        image=s1_t1.select(polarizations),
        description=f"Sentinel1_{config['site_name']}_T1",
        folder=config['export_folder'],
        fileNamePrefix=f"sentinel1_{config['site_name']}_t1",
        region=roi,
        crs=crs,
        scale=export_scale,
        maxPixels=1e12,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task_t1.start()
    tasks.append(task_t1)
    print(f"Exportacao Sentinel-1 T1 iniciada: sentinel1_{config['site_name']}_t1")

    task_t2 = ee.batch.Export.image.toDrive(
        image=s1_t2.select(polarizations),
        description=f"Sentinel1_{config['site_name']}_T2",
        folder=config['export_folder'],
        fileNamePrefix=f"sentinel1_{config['site_name']}_t2",
        region=roi,
        crs=crs,
        scale=export_scale,
        maxPixels=1e12,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task_t2.start()
    tasks.append(task_t2)
    print(f"Exportacao Sentinel-1 T2 iniciada: sentinel1_{config['site_name']}_t2")

    print(f"\nPasta de destino: {config['export_folder']}")
    return tasks


In [19]:
def export_all_sentinel_data(roi, s2_t1, s2_t2, s1_t1, s1_t2, config,
                             export_s2=True, export_s1=True,
                             s2_scale=10, s1_scale=10,
                             polarizations=['VV', 'VH']):
    """
    Exporta todos os dados Sentinel (S2 e S1) bitemporais para o Google Drive.

    Arquivos gerados:
      sentinel2_{site}_t1.tif  --  B2, B3, B4, B8, B11, B12 @ 10m
      sentinel2_{site}_t2.tif  --  B2, B3, B4, B8, B11, B12 @ 10m
      sentinel1_{site}_t1.tif  --  VV, VH normalizado [0,1] @ 10m
      sentinel1_{site}_t2.tif  --  VV, VH normalizado [0,1] @ 10m
    """
    print('\n' + '=' * 80)
    print('EXPORTACAO COMPLETA DOS DADOS SENTINEL')
    print('=' * 80)
    print(f"Site: {config['site_name']}")
    print(f"Pasta: {config['export_folder']}")
    print(f'Sentinel-2: {export_s2} | Sentinel-1: {export_s1}')
    print('=' * 80)

    all_tasks = []

    if export_s2:
        s2_tasks = export_sentinel2_bitemporal_data(roi, s2_t1, s2_t2, config, s2_scale)
        all_tasks.extend(s2_tasks)

    if export_s1:
        s1_tasks = export_sentinel1_bitemporal_data(roi, s1_t1, s1_t2, config, polarizations, s1_scale)
        all_tasks.extend(s1_tasks)

    print(f'\n{len(all_tasks)} tarefas de exportacao iniciadas!')
    print('Monitore o progresso em: https://code.earthengine.google.com/tasks')
    return all_tasks


---
# 4.0 MAIN
---

---
## 5.1 DEFINE ROI

Define the Region of Interest (ROI) by coordinates or by drawing on the interactive map.

---

In [20]:
# OPTION 1: Define by coordinates (recommended for reproducibility)
west, south, east, north = -40.331418, -3.700330, -40.305992, -3.674424
roi = ee.Geometry.Rectangle([west, south, east, north])

# OPTION 2: Define with custom Polygon (uncomment to use)
# roi = ee.Geometry.Polygon([[
#     [-40.32578, -3.69372],
#     [-40.31149, -3.69372],
#     [-40.31149, -3.67895],
#     [-40.32578, -3.67895]
# ]], None, False)

# OPTION 3: Draw interactively (execute, draw, then execute the next cell)
# Map_draw = geemap.Map()
# Map_draw.add_basemap('SATELLITE')
# display(Map_draw)

print('ROI definida com sucesso')

ROI definida com sucesso


In [21]:
# Optional: uncomment if you used OPCAO 3
# roi = Map_draw.user_roi

# View ROI and print coordinates
roi_coords = roi.coordinates().getInfo()[0]
print('Coordenadas da ROI:')
for i, coord in enumerate(roi_coords):
    print(f'  [{coord[0]:.6f}, {coord[1]:.6f}]')

Map_roi = geemap.Map()
Map_roi.centerObject(roi, 15)
Map_roi.add_basemap('SATELLITE')
Map_roi.addLayer(roi, {'color': 'yellow'}, 'ROI')
display(Map_roi)

Coordenadas da ROI:
  [-40.331418, -3.700330]
  [-40.305992, -3.700330]
  [-40.305992, -3.674424]
  [-40.331418, -3.674424]
  [-40.331418, -3.700330]


Map(center=[-3.687377027616032, -40.3187050000035], controls=(WidgetControl(options=['position', 'transparent_…

---
## 5.2 SEARCH IMAGES FOR T1

> **Note:** `COPERNICUS/S2_HARMONIZED` available from **2015-06-23**.

> Surface Reflectance (`S2_SR_HARMONIZED`) only available from **2017-03-28**.

---

In [23]:
# Search available Sentinel-2 images for T1
config_search_t1 = create_sentinel_config(
    start_date='2016-06-01',   # <-- adjust the start date
    end_date='2016-12-31',     # <-- adjust the end date
    geometry=roi,
    cloud_cover=1             # <-- Maximum cloud %
)
results_t1 = search_available_sentinel2_images(config_search_t1)

CONFIGURACAO SENTINEL
Periodo de busca: 2016-06-01 a 2016-12-31
Cobertura de nuvens maxima: 1%
Sistema de coordenadas: EPSG:32724

BUSCANDO IMAGENS SENTINEL-2: 2016-06-01 a 2016-12-31
Imagens encontradas: 2

--- IDs das imagens Sentinel-2 ---
  1. 20160904T131242_20160904T131242_T24MUB
     Data: 2016-09-04 | Nuvens: 0.09%
  2. 20160904T131242_20160904T193226_T24MUB
     Data: 2016-09-04 | Nuvens: 0.13%


In [24]:
# View candidate T1 on map (Sentinel-2 RGB + Sentinel-1 SAR)
Map_t1 = create_preview_map(
    roi,
    sentinel2_id='20160904T131242_20160904T131242_T24MUB',  # <-- change to chosen ID
    zoom=15
)
display(Map_t1)


GERANDO MAPA DE VISUALIZACAO
Carregando Sentinel-2: 20160904T131242_20160904T131242_T24MUB
  Sentinel-2 carregado
Carregando Sentinel-1 correspondente...
Processando Sentinel-1 SAR para 2016-09-04...
  Janela temporal: 2016-08-05 a 2016-10-04
  Imagens ascendentes: 0 | descendentes: 2
  Orbita selecionada: DESCENDING
  Orbitas relativas: [53, 155]
  Composicao SAR criada com sucesso
  Sentinel-1 carregado
Mapa criado com sucesso


Map(center=[-3.687377027616032, -40.3187050000035], controls=(WidgetControl(options=['position', 'transparent_…

In [25]:
# Confirm Sentinel-2 ID selected for T1
selected_sentinel2_t1 = '20160904T131242_20160904T131242_T24MUB'  # <-- adjust here
print(f'ID Sentinel-2 T1 selecionado: {selected_sentinel2_t1}')

ID Sentinel-2 T1 selecionado: 20160904T131242_20160904T131242_T24MUB


---
## 5.3 BUSCAR IMAGENS PARA T2
---

In [26]:
# Search available Sentinel-2 images for T2
config_search_t2 = create_sentinel_config(
    start_date='2023-06-01',   # <-- adjust the start date
    end_date='2023-12-31',     # <-- adjust the end date
    geometry=roi,
    cloud_cover=1            # <-- Maximum cloud %
)
results_t2 = search_available_sentinel2_images(config_search_t2)

CONFIGURACAO SENTINEL
Periodo de busca: 2023-06-01 a 2023-12-31
Cobertura de nuvens maxima: 1%
Sistema de coordenadas: EPSG:32724

BUSCANDO IMAGENS SENTINEL-2: 2023-06-01 a 2023-12-31
Imagens encontradas: 8

--- IDs das imagens Sentinel-2 ---
  1. 20230705T131249_20230705T131246_T24MUA
     Data: 2023-07-05 | Nuvens: 0.73%
  2. 20230730T131251_20230730T131247_T24MUA
     Data: 2023-07-30 | Nuvens: 0.73%
  3. 20230730T131251_20230730T131247_T24MUB
     Data: 2023-07-30 | Nuvens: 0.88%
  4. 20230824T131249_20230824T131245_T24MUA
     Data: 2023-08-24 | Nuvens: 0.20%
  5. 20231028T131241_20231028T131241_T24MUA
     Data: 2023-10-28 | Nuvens: 0.12%
  6. 20231028T131241_20231028T131241_T24MUB
     Data: 2023-10-28 | Nuvens: 0.17%
  7. 20231112T131239_20231112T131242_T24MUA
     Data: 2023-11-12 | Nuvens: 0.07%
  8. 20231112T131239_20231112T131242_T24MUB
     Data: 2023-11-12 | Nuvens: 0.70%


In [27]:
# View candidate T2 on map (Sentinel-2 RGB + Sentinel-1 SAR)
Map_t2 = create_preview_map(
    roi,
    sentinel2_id='20231112T131239_20231112T131242_T24MUA',  # <-- change to chosen ID
    zoom=15
)
display(Map_t2)


GERANDO MAPA DE VISUALIZACAO
Carregando Sentinel-2: 20231112T131239_20231112T131242_T24MUA
  Sentinel-2 carregado
Carregando Sentinel-1 correspondente...
Processando Sentinel-1 SAR para 2023-11-12...
  Janela temporal: 2023-10-13 a 2023-12-12
  Imagens ascendentes: 0 | descendentes: 5
  Orbita selecionada: DESCENDING
  Orbitas relativas: [53]
  Composicao SAR criada com sucesso
  Sentinel-1 carregado
Mapa criado com sucesso


Map(center=[-3.687377027616032, -40.3187050000035], controls=(WidgetControl(options=['position', 'transparent_…

In [28]:
# Confirm Sentinel-2 ID selected for T2
selected_sentinel2_t2 = '20231112T131239_20231112T131242_T24MUA'  # <-- ajuste aqui
print(f'ID Sentinel-2 T2 selecionado: {selected_sentinel2_t2}')

ID Sentinel-2 T2 selecionado: 20231112T131239_20231112T131242_T24MUA


---
## 5.4 PROCESS IMAGES

Loads selected Sentinel-2 images and generates Sentinel-1 composites
based on corresponding dates.

---

In [29]:
print('=' * 80)
print('PROCESSANDO IMAGENS SELECIONADAS')
print('=' * 80)

# Sentinel-2
print('\n[Sentinel-2]')
s2_t1 = get_selected_sentinel2_image(selected_sentinel2_t1, roi)
s2_t1_bands, s2_t1_proj = process_sentinel2_image(s2_t1)
print(f'  T1: {selected_sentinel2_t1}')

s2_t2 = get_selected_sentinel2_image(selected_sentinel2_t2, roi)
s2_t2_bands, s2_t2_proj = process_sentinel2_image(s2_t2)
print(f'  T2: {selected_sentinel2_t2}')

# Sentinel-1
print('\n[Sentinel-1]')
s1_t1 = generate_sentinel1_composite_from_s2_date(roi, selected_sentinel2_t1)
s1_t2 = generate_sentinel1_composite_from_s2_date(roi, selected_sentinel2_t2)

print('\nTodas as imagens processadas com sucesso!')

PROCESSANDO IMAGENS SELECIONADAS

[Sentinel-2]
Processando Sentinel-2 (imagem especifica)...
  Imagem Sentinel-2 processada
  T1: 20160904T131242_20160904T131242_T24MUB
Processando Sentinel-2 (imagem especifica)...
  Imagem Sentinel-2 processada
  T2: 20231112T131239_20231112T131242_T24MUA

[Sentinel-1]
Processando Sentinel-1 SAR para 2016-09-04...
  Janela temporal: 2016-08-05 a 2016-10-04
  Imagens ascendentes: 0 | descendentes: 2
  Orbita selecionada: DESCENDING
  Orbitas relativas: [53, 155]
  Composicao SAR criada com sucesso
Processando Sentinel-1 SAR para 2023-11-12...
  Janela temporal: 2023-10-13 a 2023-12-12
  Imagens ascendentes: 0 | descendentes: 5
  Orbita selecionada: DESCENDING
  Orbitas relativas: [53]
  Composicao SAR criada com sucesso

Todas as imagens processadas com sucesso!


---
## 5.5 CONFIGURE AND EXPORT TO GOOGLE DRIVE

The exported files will be used on the notebook:  
`02.U-net_extraction_Change_Detection_Map_for_Validation.ipynb`

---

In [30]:
# Configure bitemporal export
config_export = create_bitemporal_sentinel_config(
    t1_start_date=config_search_t1['start_date'],
    t1_end_date=config_search_t1['end_date'],
    t2_start_date=config_search_t2['start_date'],
    t2_end_date=config_search_t2['end_date'],
    geometry=roi,
    cloud_cover=10,
    sentinel2_t1_id=selected_sentinel2_t1,
    sentinel2_t2_id=selected_sentinel2_t2,
    site_name='Sobral',           # <-- name of study location
    export_crs='EPSG:32724',      # <-- export CRS
    export_folder='urban_cd_app'  # <-- folder in Google Drive
)

CONFIGURACAO BITEMPORAL SENTINEL
Periodo T1: 2016-06-01 a 2016-12-31
  Sentinel-2 T1 ID: 20160904T131242_20160904T131242_T24MUB
Periodo T2: 2023-06-01 a 2023-12-31
  Sentinel-2 T2 ID: 20231112T131239_20231112T131242_T24MUA
Cobertura de nuvens maxima: 10%
Site: Sobral
Sistema de coordenadas: EPSG:32724
Pasta de exportacao: urban_cd_app


In [31]:
# Export all data to Google Drive
tasks = export_all_sentinel_data(
    roi=roi,
    s2_t1=s2_t1,
    s2_t2=s2_t2,
    s1_t1=s1_t1,
    s1_t2=s1_t2,
    config=config_export,
    export_s2=True,          # True = exportar Sentinel-2 T1 e T2
    export_s1=True,          # True = exportar Sentinel-1 T1 e T2
    s2_scale=10,             # resolucao Sentinel-2 (metros)
    s1_scale=10,             # resolucao Sentinel-1 (metros)
    polarizations=['VV', 'VH']
)


EXPORTACAO COMPLETA DOS DADOS SENTINEL
Site: Sobral
Pasta: urban_cd_app
Sentinel-2: True | Sentinel-1: True

EXPORTANDO SENTINEL-2 PARA GOOGLE DRIVE
CRS: EPSG:32724 | Escala: 10m
Exportacao Sentinel-2 T1 iniciada: sentinel2_Sobral_t1
Exportacao Sentinel-2 T2 iniciada: sentinel2_Sobral_t2

Pasta de destino: urban_cd_app

EXPORTANDO SENTINEL-1 PARA GOOGLE DRIVE
CRS: EPSG:32724 | Polarizacoes: ['VV', 'VH'] | Escala: 10m
Exportacao Sentinel-1 T1 iniciada: sentinel1_Sobral_t1
Exportacao Sentinel-1 T2 iniciada: sentinel1_Sobral_t2

Pasta de destino: urban_cd_app

4 tarefas de exportacao iniciadas!
Monitore o progresso em: https://code.earthengine.google.com/tasks


---
## Files Generated on Google Drive

| File | Satellite | Bands / Channels | Resolution |
|---|---|---|---|
| `sentinel2_{site}_t1.tif` | Sentinel-2 | B2, B3, B4, B8, B11, B12 | 10 m |
| `sentinel2_{site}_t2.tif` | Sentinel-2 | B2, B3, B4, B8, B11, B12 | 10 m |
| `sentinel1_{site}_t1.tif` | Sentinel-1 SAR | VV, VH (norm. [0,1]) | 10 m |
| `sentinel1_{site}_t2.tif` | Sentinel-1 SAR | VV, VH (norm. [0,1]) | 10 m |

Track tasks at: https://code.earthengine.google.com/tasks

After completion, use these files on your notebook
`02.U-net_extraction_Change_Detection_Map_for_Validation.ipynb`.

---